In [7]:
import os, time, json, time, pickle, psycopg2
import boto3
import botocore
from botocore.exceptions import ClientError
from datetime import datetime

from misc import load_from_yaml, save_to_yaml
from lambdafn import build_lambda_package, print_latest_lambda_logs

from redshift_deploy import create_development_cluster
from redshift_manager import RedshiftClusterConfig, NetworkConfig, SecurityConfig, RedshiftClusterManager

from dotenv import load_dotenv

load_dotenv(os.getenv("MY_AWS_DIR", "") + "/.env")

from mylogger import CustomLogger

logger = CustomLogger()

In [2]:
ACCOUNT_ID = os.environ["AWS_ACCOUNT_ID_ROOT"]
REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

logger.info(f"VPC_ID: {ACCOUNT_ID}")

INFO: 2025-09-12 18:18:32 [1682100376.py:4] VPC_ID: 530976901147


In [ ]:
rds_client           = boto3.client('rds', region_name=REGION)
iam_client           = boto3.client('iam', region_name=REGION)
s3_client            = boto3.client('s3', region_name=REGION)
glue_client          = boto3.client('glue', region_name=REGION)
lakeformation_client = boto3.client("lakeformation", region_name=REGION)
ec2_client           = boto3.client('ec2', region_name=REGION)
ec2_resource         = boto3.resource('ec2', region_name=REGION)
events_client        = boto3.client('events', region_name=REGION)
lambda_client        = boto3.client('lambda', region_name=REGION)

# Create a CloudWatch client for Logs
logs_client = boto3.client("logs", region_name=REGION)

redshift_client = boto3.client("redshift", region_name=REGION)

-   **Notes**


-   `stl_load_errors`
    ```sql
    select * from stl_load_errors
    ```


#### Interact with Redshift Cluster using `psql`

In [ ]:
# ! telnet rsa-cluster.cxoevethw4s6.us-east-1.redshift.amazonaws.com 5439

In [11]:
# %%sh
# PGPASSWORD=$TF_VAR_redshift_master_password psql \
#   -h rsa-cluster.cxoevethw4s6.us-east-1.redshift.amazonaws.com \
#   -U $TF_VAR_redshift_master_username \
#   -d dev \
#   -p 5439

In [ ]:
redshift_master_username = os.environ["TF_VAR_redshift_master_username"]
redshift_master_password = os.environ["TF_VAR_redshift_master_password"]

! PGPASSWORD={redshift_master_password} psql \
  -h rsa-cluster.cxoevethw4s6.us-east-1.redshift.amazonaws.com \
  -U {redshift_master_username} \
  -d dev \
  -p 5439 \
  -f ./redshift_employee.sql


In [ ]:
# redshift_master_username = os.environ["TF_VAR_redshift_master_username"]
# redshift_master_password = os.environ["TF_VAR_redshift_master_password"]

# %env REDSHIFT_USER=$redshift_master_username
# %env REDSHIFT_PASS=$redshift_master_password

# ! PGPASSWORD=$REDSHIFT_PASS psql \
#   -h rsa-cluster.cxoevethw4s6.us-east-1.redshift.amazonaws.com \
#   -U $REDSHIFT_USER \
#   -d dev \
#   -p 5439 \
#   -f ./redshift_employee.sql


#### Interact with Redshift Cluster using `psycopg2` driver

In [12]:
host_name = "rsa-cluster.cxoevethw4s6.us-east-1.redshift.amazonaws.com"

In [ ]:
conn = psycopg2.connect(
    host=host_name,
    port=5439,
    user=os.environ["TF_VAR_redshift_master_username"],
    password=os.environ["TF_VAR_redshift_master_password"],
    database="dev"
)

In [6]:
cur = conn.cursor()